In [14]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import json
import time

In [15]:
START_URLS = {
    "rekrutacja": [
        "https://kandydacipb.edu.pl/studia-i-stopnia/rekrutacja-krok-po-kroku-studia-i-stopnia/",
        "https://kandydacipb.edu.pl/rekrutacja/studia-ii-stopnia/rekrutacja-krok-po-kroku-studia-ii-stopnia",
        "https://kandydacipb.edu.pl/rekrutacja/",
        "https://kandydacipb.edu.pl/faq/",
    ],
    "studia": [
        "https://kandydacipb.edu.pl/kierunki-studiow/",
        "https://kandydacipb.edu.pl/studia-i-stopnia/",
        "https://kandydacipb.edu.pl/studia-ii-stopnia/",
    ],
    "kontakt": [
        "https://pb.edu.pl/kontakt/dane-teleadresowe/",
        "https://kandydacipb.edu.pl/kontakt/",
    ],
    "stypendia": [
        "https://pb.edu.pl/studenci/stypendia/",
    ],
    "akademik": [
        "https://pb.edu.pl/studenci/akademiki-pb/",
    ],
}

HEADERS = {
    "User-Agent": "StudentAssistantBot/1.0"
}

ALLOWED_DOMAINS = {
    "pb.edu.pl",
    "kandydacipb.edu.pl",
    # "irk.pb.edu.pl",
    # "irk2.uci.pb.edu.pl",
    # "akademiki.bialystok.pl",
}

In [16]:
ALLOW_PATTERNS = {
    "rekrutacja": [
        "kandydacipb.edu.pl/rekrutacja",
        "kandydacipb.edu.pl/studia-i-stopnia/rekrutacja-krok-po-kroku",
        "kandydacipb.edu.pl/rekrutacja/studia-ii-stopnia/rekrutacja-krok-po-kroku",
        "kandydacipb.edu.pl/faq",
    ],
    "studia": [
        "kandydacipb.edu.pl/kierunki-studiow",
        "kandydacipb.edu.pl/studia-i-stopnia",
        "kandydacipb.edu.pl/studia-ii-stopnia",
    ],
    "kontakt": [
        "pb.edu.pl/kontakt/dane-teleadresowe",
        "kandydacipb.edu.pl/kontakt",
    ],
    "stypendia": [
        "pb.edu.pl/studenci/stypendia",
    ],
    "akademik": [
        "pb.edu.pl/studenci/akademiki-pb",
    ],
}

def is_useful_url(url, category):
    url = url.lower()
    return any(pattern in url for pattern in ALLOW_PATTERNS.get(category, []))

In [17]:
SKIP_WORDS = [
    "aktualnosci", "news", "wydarzenia", "kalendarz",
    "tag", "category", "author", "page/",
    "nggallery", "galeria", "wp-content",
    "polityka-prywatnosci", "cookies",
    ".jpg", ".jpeg", ".png", ".webp", ".svg", ".pdf",
    "facebook", "instagram", "youtube", "linkedin",
    "mailto:", "tel:",
    "bialjam", "juwenalia", "festiwal", "konkurs",
    "wosp", "targi", "piknik", "konferencja",
    "kolo-naukowe",
    "patent", "ekoskora", "rover", "zawod-inzynier",
    "obsluga-informatyczna", "faq-obsluga-informatyczna", "redakcja-serwisu-www", "mapa-social-media",
]

def should_skip(url):
    url = url.lower()
    return any(word in url for word in SKIP_WORDS)

In [18]:
BAD_MARKERS = [
    "Ustawienia ciasteczek",
    "Ta strona używa ciasteczek",
    "W ramach naszego serwisu www stosujemy pliki cookies",
    "Używamy plików cookie",
    "Zamknij ustawienia ciasteczek RODO",
    "Polityka prywatności Niezbędne pliki cookie",
    "Powered by Zgodności ciasteczek z RODO",
]

def remove_cookie_text(text):
    for marker in BAD_MARKERS:
        idx = text.find(marker)
        if idx != -1:
            text = text[:idx]
    return text.strip()

In [19]:
def is_valid_url(url):
    parsed = urlparse(url)
    if parsed.scheme not in ['https', 'http']:
        return False
    domain = parsed.netloc.lower().removeprefix("www.")
    return domain in ALLOWED_DOMAINS

In [20]:
def clean_text(soup, url):
    tags_to_remove = ["script", "style", "nav", "footer", "header", "form", "aside"]

    if url.rstrip("/") == "https://kandydacipb.edu.pl/rekrutacja":
        for article in soup.select("article.et_pb_post, article.post"):
            article.decompose()

    for tag in soup(tags_to_remove):
        tag.decompose()

    text = soup.get_text(separator=" ")
    text = " ".join(text.split())
    return text

In [21]:
def get_title(soup):
    if soup.title:
        return soup.title.get_text(strip=True)
    return ""

In [22]:
def get_links(soup, url):
    if url.rstrip("/") == "https://kandydacipb.edu.pl/rekrutacja":
        for article in soup.select("article.et_pb_post, article.post"):
            article.decompose()

    return [
        urljoin(url, a["href"])
        for a in soup.find_all("a", href=True)
    ]

In [23]:
def scrape_page(url):
    response = requests.get(url, headers=HEADERS, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    soup_for_links = BeautifulSoup(response.text, "html.parser")

    links = get_links(soup_for_links, url)
    text = clean_text(soup, url)
    text = remove_cookie_text(text)

    return {
        "title": get_title(soup),
        "text": text,
        "links": links,
    }

In [24]:
def crawl_category(category, start_urls, max_depth=2):
    visited = set()
    results = []
    queue = [(url, 0) for url in start_urls]

    while queue:
        url, depth = queue.pop(0)

        if url in visited or depth > max_depth:
            continue

        visited.add(url)

        try:
            page = scrape_page(url)
        except Exception as e:
            print(f"Error: {url} -> {e}")
            continue

        if len(page['text']) > 300:
            results.append({
                "url": url,
                "title": page['title'],
                'text': page['text'],
                'category': category,
            })
        
        for link in page['links']:
            link = link.split('#')[0].rstrip("/")

            if should_skip(link):
                continue

            if not is_valid_url(link):
                continue

            if not is_useful_url(link, category):
                continue

            if link not in visited:
                queue.append((link, depth + 1))
            
        time.sleep(0.5)

    return results

#### *Scrape trwa: ~2m*

In [25]:
all_pages = []
category_counts = {}

for category, urls in START_URLS.items():
    if category in ["kontakt", 'akademik', 'stypendia']:
        depth = 0
    else:
        depth = 1

    pages = crawl_category(category, urls, max_depth=depth)
    category_counts[category] = len(pages)
    all_pages.extend(pages)

unique_pages = {}

for page in all_pages:
    url = page["url"].rstrip("/")

    if url not in unique_pages:
        page["url"] = url
        unique_pages[url] = page

all_pages = list(unique_pages.values())

with open("data/pages.jsonl", "w", encoding="utf-8") as f:
    for page in all_pages:
        f.write(json.dumps(page, ensure_ascii=False) + "\n")

print(f"Zapisano stron łącznie: {len(all_pages)}")
for category, count in category_counts.items():
    print(f"Zapisano stron ({category}): {count}")

Error: https://kandydacipb.edu.pl/kierunki-studiow/{{study.link.url}} -> 404 Client Error: Not Found for url: https://kandydacipb.edu.pl/kierunki-studiow/%7B%7Bstudy.link.url
Zapisano stron łącznie: 44
Zapisano stron (rekrutacja): 30
Zapisano stron (studia): 17
Zapisano stron (kontakt): 2
Zapisano stron (stypendia): 1
Zapisano stron (akademik): 1


In [26]:
all_pages

[{'url': 'https://kandydacipb.edu.pl/studia-i-stopnia/rekrutacja-krok-po-kroku-studia-i-stopnia',
  'title': 'Rekrutacja krok po kroku – studia I stopnia - Politechnika Białostocka',
  'text': 'Rekrutacja krok po kroku – studia I stopnia - Politechnika Białostocka Facebook Instagram youtube linkedin tiktok Strona główna 9 Studia I stopnia 9 Rekrutacja krok po kroku – studia I stopnia Studia I stopnia \uf002 Wyszukaj kierunek l Aplikuj online \ue0e7 Oblicz wzór rekrutacyjny Zapoznaj się z ofertą kierunków Znajdź kierunek Zarejestruj się w systemie IRK Internetowa Rejestracja Kandydatów Po zalogowaniu się do systemu IRK Wprowadź dane osobowe. Zapisz się na wybrany kierunek/kierunki studiów Wnieś opłatę rekrutacyjną na wygenerowany w IRK indywidualny numer konta bankowego Sprawdź wysokość opłaty rekrutacyjnej Kolejne kroki wykonaj zgodnie z harmonogramem rekrutacji Przejdź do harmonogramu rekrutacji Ulotka informacyjna',
  'category': 'rekrutacja'},
 {'url': 'https://kandydacipb.edu.pl/re